# Touch Algo V1 Validation (Converted from .py)
该 Notebook 由 build/touch_algo_v1_validate.py 转换而来，可直接在 Colab/本地 Notebook 运行。

In [23]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import argparse
import configparser
import math
import random
from collections import deque, defaultdict
from typing import List, Tuple, Dict

ROWS = 40
COLS = 60
DIR8 = [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]


@dataclass
class AlgoConfig:
    base_threshold: int = 150
    min_blob_area: int = 3
    peak_ratio: float = 0.35
    min_peak_abs: int = 250
    min_peak_distance: float = 2.0
    valley_ratio_split: float = 0.82
    fat_area_min: int = 18
    fat_peak_uniqueness: float = 0.42
    fat_close_peak_dist: float = 3.2
    fat_valley_keep_ratio: float = 0.72
    axis_seed_aspect_min: float = 1.8
    axis_seed_min_rel_height: float = 0.28
    split_var_threshold: float = 450000.0
    split_aspect_threshold: float = 1.6
    peel_aspect_min: float = 1.9
    peel_var_threshold: float = 500000.0
    peel_floor_ratio: float = 0.20
    peel_subtract_scale: float = 0.78
    tri_hint_min_blob_area: int = 14
    tri_hint_aspect_min: float = 1.5
    tri_hint_aspect_max: float = 8.0
    contact_merge_dist: float = 0.8


@dataclass
class Contact:
    x: float
    y: float
    strength: float
    route: str


def load_algo_config(config_path: Path) -> AlgoConfig:
    cfg = AlgoConfig()
    parser = configparser.ConfigParser()
    parser.read(config_path, encoding="utf-8")

    if parser.has_section("Signal Segmenter (Phase 1)"):
        sec = parser["Signal Segmenter (Phase 1)"]
        cfg.base_threshold = int(sec.get("BaseThreshold", cfg.base_threshold))

    # 这些是算法 V1 的 Python 侧参数，后续你审核后再落地到 C++ UI
    return cfg


def parse_dvr_csv(path: Path) -> List[List[List[int]]]:
    lines = path.read_text(encoding="utf-8").splitlines()
    frames: List[List[List[int]]] = []
    i = 0
    while i < len(lines):
        if lines[i].startswith("--- Frame"):
            while i < len(lines) and lines[i].strip() != "Heatmap:":
                i += 1
            if i >= len(lines):
                break
            i += 1
            if i + ROWS > len(lines):
                break

            mat: List[List[int]] = []
            ok = True
            for r in range(ROWS):
                parts = lines[i + r].strip().split(",")
                if len(parts) != COLS:
                    ok = False
                    break
                try:
                    mat.append([int(x) for x in parts])
                except ValueError:
                    ok = False
                    break
            if ok:
                frames.append(mat)
            i += ROWS
        else:
            i += 1
    return frames


def neighbors8(y: int, x: int):
    for dy, dx in DIR8:
        ny, nx = y + dy, x + dx
        if 0 <= ny < ROWS and 0 <= nx < COLS:
            yield ny, nx


def find_blobs(mat: List[List[int]], threshold: int) -> List[List[Tuple[int, int, int]]]:
    visited = [[False] * COLS for _ in range(ROWS)]
    blobs: List[List[Tuple[int, int, int]]] = []

    for y in range(ROWS):
        for x in range(COLS):
            if visited[y][x] or mat[y][x] < threshold:
                continue
            q = deque([(y, x)])
            visited[y][x] = True
            blob: List[Tuple[int, int, int]] = []
            while q:
                cy, cx = q.popleft()
                blob.append((cy, cx, mat[cy][cx]))
                for ny, nx in neighbors8(cy, cx):
                    if not visited[ny][nx] and mat[ny][nx] >= threshold:
                        visited[ny][nx] = True
                        q.append((ny, nx))
            blobs.append(blob)
    return blobs


def local_maxima(blob: List[Tuple[int, int, int]], mat: List[List[int]], cfg: AlgoConfig) -> List[Tuple[int, int, int]]:
    blob_set = {(y, x) for y, x, _ in blob}
    peak_floor = max(cfg.min_peak_abs, int(max(v for _, _, v in blob) * cfg.peak_ratio))
    peaks: List[Tuple[int, int, int]] = []

    for y, x, v in blob:
        if v < peak_floor:
            continue
        is_peak = True
        for ny, nx in neighbors8(y, x):
            if (ny, nx) in blob_set and mat[ny][nx] > v:
                is_peak = False
                break
        if is_peak:
            peaks.append((y, x, v))

    peaks.sort(key=lambda p: p[2], reverse=True)

    # NMS by distance
    filtered: List[Tuple[int, int, int]] = []
    dist2 = cfg.min_peak_distance * cfg.min_peak_distance
    for p in peaks:
        py, px, _ = p
        keep = True
        for q in filtered:
            qy, qx, _ = q
            if (py - qy) * (py - qy) + (px - qx) * (px - qx) < dist2:
                keep = False
                break
        if keep:
            filtered.append(p)
    return filtered


def weighted_centroid(points: List[Tuple[int, int, int]]) -> Tuple[float, float, float]:
    sw = 0.0
    sx = 0.0
    sy = 0.0
    for y, x, v in points:
        w = float(max(0, v))
        sw += w
        sx += x * w
        sy += y * w
    if sw <= 1e-6:
        return 0.0, 0.0, 0.0
    return sx / sw, sy / sw, sw


def blob_intensity_variance(blob: List[Tuple[int, int, int]]) -> float:
    if len(blob) <= 1:
        return 0.0
    vals = [v for _, _, v in blob]
    mean = sum(vals) / len(vals)
    return sum((v - mean) * (v - mean) for v in vals) / len(vals)


def blob_aspect_ratio(blob: List[Tuple[int, int, int]]) -> float:
    sw = sum(max(0, v) for _, _, v in blob)
    if sw <= 1e-6:
        return 1.0

    mx = sum(x * v for y, x, v in blob) / sw
    my = sum(y * v for y, x, v in blob) / sw

    cxx = sum(v * (x - mx) * (x - mx) for y, x, v in blob) / sw
    cyy = sum(v * (y - my) * (y - my) for y, x, v in blob) / sw
    cxy = sum(v * (x - mx) * (y - my) for y, x, v in blob) / sw

    tr = cxx + cyy
    det = cxx * cyy - cxy * cxy
    disc = max(0.0, tr * tr - 4.0 * det)
    l1 = 0.5 * (tr + math.sqrt(disc))
    l2 = max(1e-6, 0.5 * (tr - math.sqrt(disc)))
    return l1 / l2


def blob_major_axis(blob: List[Tuple[int, int, int]]) -> Tuple[float, float, float, float]:
    sw = sum(max(0, v) for _, _, v in blob)
    if sw <= 1e-6:
        return 1.0, 0.0, 0.0, 0.0

    mx = sum(x * v for y, x, v in blob) / sw
    my = sum(y * v for y, x, v in blob) / sw

    cxx = sum(v * (x - mx) * (x - mx) for y, x, v in blob) / sw
    cyy = sum(v * (y - my) * (y - my) for y, x, v in blob) / sw
    cxy = sum(v * (x - mx) * (y - my) for y, x, v in blob) / sw

    tr = cxx + cyy
    det = cxx * cyy - cxy * cxy
    disc = max(0.0, tr * tr - 4.0 * det)
    l1 = 0.5 * (tr + math.sqrt(disc))

    vx = cxy
    vy = l1 - cxx
    norm = math.sqrt(vx * vx + vy * vy)
    if norm <= 1e-6:
        vx, vy = 1.0, 0.0
    else:
        vx /= norm
        vy /= norm
    return vx, vy, mx, my


def axis_profile_seeds(
    blob: List[Tuple[int, int, int]],
    min_rel_height: float,
) -> List[Tuple[int, int, int]]:
    if len(blob) < 6:
        return []

    vx, vy, mx, my = blob_major_axis(blob)
    bins: Dict[int, float] = defaultdict(float)

    for y, x, v in blob:
        t = (x - mx) * vx + (y - my) * vy
        key = int(round(t * 2.0))
        bins[key] += float(v)

    if not bins:
        return []

    keys = sorted(bins.keys())
    arr = [bins[k] for k in keys]
    if len(arr) < 3:
        return []

    sm = arr[:]
    for i in range(1, len(arr) - 1):
        sm[i] = 0.25 * arr[i - 1] + 0.5 * arr[i] + 0.25 * arr[i + 1]

    h_floor = max(sm) * min_rel_height
    seeds: List[Tuple[int, int, int]] = []
    for i in range(1, len(sm) - 1):
        if sm[i] >= h_floor and sm[i] >= sm[i - 1] and sm[i] >= sm[i + 1]:
            t = keys[i] / 2.0
            sx = int(round(mx + t * vx))
            sy = int(round(my + t * vy))
            sx = max(0, min(COLS - 1, sx))
            sy = max(0, min(ROWS - 1, sy))
            peak_v = int(max(1.0, sm[i] / 4.0))
            seeds.append((sy, sx, peak_v))
    return seeds


def merge_peaks_by_distance(
    peaks: List[Tuple[int, int, int]],
    extra: List[Tuple[int, int, int]],
    min_dist: float,
) -> List[Tuple[int, int, int]]:
    merged = list(peaks)
    d2 = min_dist * min_dist
    for py, px, pv in extra:
        keep = True
        for qy, qx, _ in merged:
            if (py - qy) * (py - qy) + (px - qx) * (px - qx) < d2:
                keep = False
                break
        if keep:
            merged.append((py, px, pv))
    merged.sort(key=lambda p: p[2], reverse=True)
    return merged


def contacts_from_peak_seeds(
    blob: List[Tuple[int, int, int]],
    seeds: List[Tuple[int, int, int]],
    max_contacts: int = 3,
) -> List[Tuple[float, float, float]]:
    blob_map = {(y, x): v for y, x, v in blob}
    out: List[Tuple[float, float, float]] = []
    for sy, sx, _ in seeds[:max_contacts]:
        sw = 0.0
        sxw = 0.0
        syw = 0.0
        for dy in (-1, 0, 1):
            for dx in (-1, 0, 1):
                ny, nx = sy + dy, sx + dx
                w = float(blob_map.get((ny, nx), 0))
                if w <= 0:
                    continue
                sw += w
                sxw += nx * w
                syw += ny * w
        if sw > 1e-6:
            out.append((sxw / sw, syw / sw, sw))
    return out


def line_min_val(mat: List[List[int]], y1: int, x1: int, y2: int, x2: int) -> int:
    steps = max(abs(y2 - y1), abs(x2 - x1), 1)
    min_v = 10**9
    for s in range(steps + 1):
        t = s / steps
        y = int(round(y1 + t * (y2 - y1)))
        x = int(round(x1 + t * (x2 - x1)))
        y = max(0, min(ROWS - 1, y))
        x = max(0, min(COLS - 1, x))
        min_v = min(min_v, mat[y][x])
    return min_v


def is_fat_finger(blob: List[Tuple[int, int, int]], peaks: List[Tuple[int, int, int]], mat: List[List[int]], cfg: AlgoConfig) -> bool:
    area = len(blob)
    aspect = blob_aspect_ratio(blob)

    if area < cfg.fat_area_min:
        return False
    if len(peaks) <= 1:
        return True

    p1 = peaks[0][2]
    p2 = peaks[1][2]
    if p1 <= 0:
        return False

    y1, x1, _ = peaks[0]
    y2, x2, _ = peaks[1]
    dist = math.sqrt((y1 - y2) * (y1 - y2) + (x1 - x2) * (x1 - x2))
    valley = line_min_val(mat, y1, x1, y2, x2)
    valley_ratio = valley / max(1, min(p1, p2))

    # 双峰很近且谷不够深，优先视为单指重压
    if dist <= cfg.fat_close_peak_dist and valley_ratio >= cfg.fat_valley_keep_ratio:
        return True

    # 超大面积下，允许三个以内近峰也归并为单指
    if area >= int(cfg.fat_area_min * 1.8) and len(peaks) <= 3 and valley_ratio >= 0.62:
        return True

    # 第二峰相对第一峰过弱 -> 更像单指大面积压下
    return (p2 / p1) < cfg.fat_peak_uniqueness


def should_split(
    peaks: List[Tuple[int, int, int]],
    mat: List[List[int]],
    cfg: AlgoConfig,
    blob: List[Tuple[int, int, int]],
) -> bool:
    if len(peaks) < 2:
        return False

    # 无时域策略下的方差驱动分裂：3个稳定峰直接允许分裂
    if len(peaks) >= 3:
        return True

    # 任意两个强峰存在明显谷底，才分裂
    for i in range(min(3, len(peaks))):
        for j in range(i + 1, min(3, len(peaks))):
            y1, x1, v1 = peaks[i]
            y2, x2, v2 = peaks[j]
            valley = line_min_val(mat, y1, x1, y2, x2)
            min_peak = min(v1, v2)
            if min_peak <= 0:
                continue
            ratio = valley / min_peak
            if ratio <= cfg.valley_ratio_split:
                return True

    # 谷底不明显时，用空域方差和形状方差兜底
    variance = blob_intensity_variance(blob)
    aspect = blob_aspect_ratio(blob)
    if len(peaks) >= 2 and variance > cfg.split_var_threshold and aspect > cfg.split_aspect_threshold:
        return True

    return False


def split_blob_by_peaks(blob: List[Tuple[int, int, int]], peaks: List[Tuple[int, int, int]], keep_k: int = 3) -> List[List[Tuple[int, int, int]]]:
    seeds = peaks[:keep_k]
    groups: Dict[int, List[Tuple[int, int, int]]] = defaultdict(list)
    for y, x, v in blob:
        best_id = 0
        best_score = -1e18
        for idx, (py, px, pv) in enumerate(seeds):
            d2 = (y - py) * (y - py) + (x - px) * (x - px)
            # 简化的“峰值引导距离”打分
            score = pv - 80.0 * math.sqrt(d2)
            if score > best_score:
                best_score = score
                best_id = idx
        groups[best_id].append((y, x, v))
    return list(groups.values())


def residual_peak_peeling(
    blob: List[Tuple[int, int, int]],
    cfg: AlgoConfig,
    max_contacts: int = 3,
) -> List[Tuple[float, float, float]]:
    residual = {(y, x): float(v) for y, x, v in blob}
    if not residual:
        return []

    blob_max = max(residual.values())
    floor = max(float(cfg.min_peak_abs), cfg.peel_floor_ratio * blob_max)

    picked: List[Tuple[float, float, float]] = []
    sigma2 = 1.2 * 1.2
    subtract_scale = cfg.peel_subtract_scale

    for _ in range(max_contacts):
        (py, px), pv = max(residual.items(), key=lambda kv: kv[1])
        if pv < floor:
            break

        sw = 0.0
        sx = 0.0
        sy = 0.0
        for dy in (-1, 0, 1):
            for dx in (-1, 0, 1):
                ny, nx = py + dy, px + dx
                w = residual.get((ny, nx), 0.0)
                if w <= 0:
                    continue
                sw += w
                sx += nx * w
                sy += ny * w
        if sw <= 1e-6:
            break

        cx = sx / sw
        cy = sy / sw
        picked.append((cx, cy, sw))

        # 减去一个局部高斯核，释放肩峰
        for yy in range(py - 2, py + 3):
            for xx in range(px - 2, px + 3):
                if (yy, xx) not in residual:
                    continue
                d2 = (yy - py) * (yy - py) + (xx - px) * (xx - px)
                g = math.exp(-d2 / (2.0 * sigma2))
                residual[(yy, xx)] = max(0.0, residual[(yy, xx)] - pv * subtract_scale * g)

    return picked


def merge_close_contacts(
    contacts: List[Contact],
    min_dist: float = 1.0,
) -> List[Contact]:
    if len(contacts) <= 1:
        return contacts

    merged = contacts[:]
    min_d2 = min_dist * min_dist

    changed = True
    while changed and len(merged) > 1:
        changed = False
        best_i, best_j = -1, -1
        best_d2 = 1e18
        for i in range(len(merged)):
            for j in range(i + 1, len(merged)):
                dx = merged[i].x - merged[j].x
                dy = merged[i].y - merged[j].y
                d2 = dx * dx + dy * dy
                if d2 < best_d2:
                    best_d2 = d2
                    best_i, best_j = i, j

        if best_d2 < min_d2 and best_i >= 0 and best_j >= 0:
            a = merged[best_i]
            b = merged[best_j]
            sw = max(1e-6, a.strength + b.strength)
            nx = (a.x * a.strength + b.x * b.strength) / sw
            ny = (a.y * a.strength + b.y * b.strength) / sw
            ns = a.strength + b.strength
            new_contact = Contact(x=nx, y=ny, strength=ns, route="merged_split")

            keep = [c for k, c in enumerate(merged) if k not in (best_i, best_j)]
            keep.append(new_contact)
            merged = keep
            changed = True

    return merged


def parse_frame(mat: List[List[int]], cfg: AlgoConfig) -> List[Contact]:
    contacts: List[Contact] = []
    blobs = [b for b in find_blobs(mat, cfg.base_threshold) if len(b) >= cfg.min_blob_area]

    for blob in blobs:
        blob_contacts: List[Contact] = []
        peaks = local_maxima(blob, mat, cfg)
        aspect = blob_aspect_ratio(blob)

        # 对并拢/山脊型 blob 增加主轴轮廓峰，提升双指/三指召回
        if aspect > cfg.axis_seed_aspect_min and len(peaks) < 3:
            axis_seeds = axis_profile_seeds(blob, min_rel_height=cfg.axis_seed_min_rel_height)
            peaks = merge_peaks_by_distance(peaks, axis_seeds, cfg.min_peak_distance)

        if is_fat_finger(blob, peaks, mat, cfg):
            x, y, s = weighted_centroid(blob)
            blob_contacts.append(Contact(x=x, y=y, strength=s, route="fat_finger_single"))
            contacts.extend(blob_contacts)
            continue

        if should_split(peaks, mat, cfg, blob):
            variance = blob_intensity_variance(blob)
            has_tri_hint = (
                len(peaks) >= 3
                and len(blob) >= cfg.tri_hint_min_blob_area
                and aspect > cfg.tri_hint_aspect_min
                and aspect < cfg.tri_hint_aspect_max
            )

            # 方差主导场景先尝试残差剥离（对重叠三指更友好）
            if aspect > cfg.peel_aspect_min and variance > cfg.peel_var_threshold:
                peeled = residual_peak_peeling(blob, cfg, max_contacts=3)
                if has_tri_hint and len(peeled) < 3:
                    seeded = contacts_from_peak_seeds(blob, peaks, max_contacts=3)
                    if len(seeded) >= 3:
                        peeled = seeded
                if len(peeled) >= 2:
                    for x, y, s in peeled:
                        blob_contacts.append(Contact(x=x, y=y, strength=s, route="merged_split"))
                    blob_contacts = merge_close_contacts(blob_contacts, min_dist=cfg.contact_merge_dist)
                    contacts.extend(blob_contacts)
                    continue

            sub_blobs = split_blob_by_peaks(blob, peaks)
            for sb in sub_blobs:
                if len(sb) < cfg.min_blob_area:
                    continue
                x, y, s = weighted_centroid(sb)
                blob_contacts.append(Contact(x=x, y=y, strength=s, route="merged_split"))
            if has_tri_hint and len(blob_contacts) < 3:
                seeded = contacts_from_peak_seeds(blob, peaks, max_contacts=3)
                if len(seeded) >= 3:
                    blob_contacts = [Contact(x=x, y=y, strength=s, route="merged_split") for x, y, s in seeded]

            blob_contacts = merge_close_contacts(blob_contacts, min_dist=cfg.contact_merge_dist)
            contacts.extend(blob_contacts)
            continue

        x, y, s = weighted_centroid(blob)
        blob_contacts.append(Contact(x=x, y=y, strength=s, route="normal_single"))
        contacts.extend(blob_contacts)

    return contacts


def run_dataset(dvr_path: Path, cfg: AlgoConfig) -> Dict[str, float]:
    frames = parse_dvr_csv(dvr_path)
    total_contacts = 0
    active_frames = 0
    route_count = defaultdict(int)
    active_contact_counts: List[int] = []

    target_contacts = infer_target_contacts(dvr_path.name)

    for mat in frames:
        peak = max(max(row) for row in mat)
        is_active = peak >= cfg.base_threshold
        if is_active:
            active_frames += 1

        contacts = parse_frame(mat, cfg)
        contact_count = len(contacts)
        total_contacts += contact_count
        if is_active:
            active_contact_counts.append(contact_count)
        for c in contacts:
            route_count[c.route] += 1

    avg_contacts = (total_contacts / active_frames) if active_frames else 0.0
    variance = 0.0
    mse = 0.0
    if active_contact_counts:
        variance = sum((c - avg_contacts) ** 2 for c in active_contact_counts) / len(active_contact_counts)
        if target_contacts is not None:
            mse = sum((c - target_contacts) ** 2 for c in active_contact_counts) / len(active_contact_counts)

    out = {
        "frames": len(frames),
        "active_frames": active_frames,
        "target_contacts": target_contacts if target_contacts is not None else -1,
        "avg_contacts_per_active": round(avg_contacts, 3),
        "contacts_variance": round(variance, 3),
        "contacts_mse": round(mse, 3),
    }
    for k, v in route_count.items():
        out[f"route_{k}"] = v
    return out


def infer_target_contacts(file_name: str) -> int | None:
    lower = file_name.lower()
    if "3finger" in lower:
        return 3
    if "backtrack" in lower or "2finger" in lower:
        return 2
    if "singlehuge" in lower or "single" in lower:
        return 1
    return None


def score_results(results: Dict[str, Dict[str, float]]) -> float:
    total = 0.0
    for name, r in results.items():
        target = int(r.get("target_contacts", -1))
        avg = float(r.get("avg_contacts_per_active", 0.0))
        var = float(r.get("contacts_variance", 0.0))
        mse = float(r.get("contacts_mse", 0.0))

        if target == 3:
            w = 1.2
        elif target == 2:
            w = 1.0
        elif target == 1:
            w = 0.35
        else:
            w = 0.2

        total += w * (2.2 * mse + 2.8 * abs(avg - max(target, 0)) + 0.25 * var)
    return total


def sample_candidate(base_cfg: AlgoConfig, rng: random.Random) -> AlgoConfig:
    return AlgoConfig(
        base_threshold=base_cfg.base_threshold,
        min_blob_area=base_cfg.min_blob_area,
        peak_ratio=round(rng.uniform(0.30, 0.43), 3),
        min_peak_abs=base_cfg.min_peak_abs,
        min_peak_distance=round(rng.uniform(1.7, 2.5), 2),
        valley_ratio_split=round(rng.uniform(0.76, 0.88), 3),
        fat_area_min=base_cfg.fat_area_min,
        fat_peak_uniqueness=round(rng.uniform(0.36, 0.50), 3),
        fat_close_peak_dist=round(rng.uniform(2.8, 3.8), 2),
        fat_valley_keep_ratio=round(rng.uniform(0.66, 0.80), 3),
        axis_seed_aspect_min=round(rng.uniform(1.6, 2.2), 2),
        axis_seed_min_rel_height=round(rng.uniform(0.22, 0.36), 3),
        split_var_threshold=round(rng.uniform(320000.0, 620000.0), 1),
        split_aspect_threshold=round(rng.uniform(1.35, 1.9), 2),
        peel_aspect_min=round(rng.uniform(1.7, 2.3), 2),
        peel_var_threshold=round(rng.uniform(360000.0, 700000.0), 1),
        peel_floor_ratio=round(rng.uniform(0.14, 0.30), 3),
        peel_subtract_scale=round(rng.uniform(0.64, 0.88), 3),
        tri_hint_min_blob_area=base_cfg.tri_hint_min_blob_area,
        tri_hint_aspect_min=base_cfg.tri_hint_aspect_min,
        tri_hint_aspect_max=base_cfg.tri_hint_aspect_max,
        contact_merge_dist=round(rng.uniform(0.65, 1.05), 2),
    )


def optimize_parameters(files: List[Path], base_cfg: AlgoConfig, trials: int = 180, topk: int = 8) -> tuple[AlgoConfig, Dict[str, Dict[str, float]], float]:
    rng = random.Random(20260301)
    candidates: List[tuple[float, AlgoConfig, Dict[str, Dict[str, float]]]] = []

    baseline_results = {p.name: run_dataset(p, base_cfg) for p in files}
    baseline_score = score_results(baseline_results)
    candidates.append((baseline_score, base_cfg, baseline_results))

    for _ in range(trials):
        cand = sample_candidate(base_cfg, rng)
        cand_results = {p.name: run_dataset(p, cand) for p in files}
        cand_score = score_results(cand_results)
        candidates.append((cand_score, cand, cand_results))

    candidates.sort(key=lambda x: x[0])
    finalists = candidates[:max(1, topk)]
    finalists.sort(key=lambda x: x[0])
    best_score, best_cfg, best_results = finalists[0]
    return best_cfg, best_results, best_score


def main():
    parser = argparse.ArgumentParser(description="Touch parsing validator and optimizer")
    parser.add_argument("--optimize", action="store_true", help="run random search by mean/variance/MSE")
    parser.add_argument("--trials", type=int, default=180, help="random search trials")
    parser.add_argument("--topk", type=int, default=8, help="top candidate count kept")
    args, _unknown = parser.parse_known_args()

    try:
        root = Path(__file__).resolve().parents[1]
    except NameError:
        root = Path.cwd().resolve().parent
    config_path = root / "build" / "config.ini"
    dvr_dir = root / "build" / "exports" / "dvr"

    cfg = load_algo_config(config_path)
    files = sorted([p for p in dvr_dir.glob("*.csv")])

    print("=== Touch Algo V1 Validation ===")
    print(f"config: {config_path}")
    print(f"base_threshold={cfg.base_threshold}")
    print(f"fat_area_min={cfg.fat_area_min}, valley_ratio_split={cfg.valley_ratio_split}")
    print()

    if args.optimize:
        best_cfg, best_results, best_score = optimize_parameters(files, cfg, trials=args.trials, topk=args.topk)
        print("=== Optimized Parameters ===")
        print(best_cfg)
        print(f"objective_score={best_score:.4f}")
        print()
        for name, result in best_results.items():
            print(f"[{name}]")
            for k, v in result.items():
                print(f"  {k}: {v}")
            print()
        return

    for p in files:
        result = run_dataset(p, cfg)
        print(f"[{p.name}]")
        for k, v in result.items():
            print(f"  {k}: {v}")
        print()


# In notebook/colab, call main() in the next cell manually.


In [24]:
# Baseline run
main()

# Optimization run example
# main_args = ["--optimize", "--trials", "180", "--topk", "8"]
# import sys
# _old = sys.argv
# sys.argv = ["touch_algo_v1_validate.py"] + main_args
# main()
# sys.argv = _old

=== Touch Algo V1 Validation ===
config: /build/config.ini
base_threshold=150
fat_area_min=18, valley_ratio_split=0.82



In [25]:
import os
from pathlib import Path

workspace_root = Path(r"d:/source/repos/EGoTouchRev-rebuild")
if workspace_root.exists():
    os.chdir(workspace_root)

print("cwd:", Path.cwd())
print("config exists:", (Path.cwd() / "build" / "config.ini").exists())
print("dvr exists:", (Path.cwd() / "build" / "exports" / "dvr").exists())

main()

cwd: /content
config exists: False
dvr exists: False
=== Touch Algo V1 Validation ===
config: /build/config.ini
base_threshold=150
fat_area_min=18, valley_ratio_split=0.82



In [26]:
from pathlib import Path

candidates = list(Path('/content').glob('**/build/config.ini'))
print('found config candidates:', len(candidates))
for p in candidates[:10]:
    print(p)

if candidates:
    root = candidates[0].parents[1]
    import os
    os.chdir(root)
    print('auto switched cwd to:', Path.cwd())
    main()

found config candidates: 0


## 无手动上传（Colab VSCode 插件）
本单元支持纯 Python 方式导入项目，不需要 `files.upload()` 文件选择框。

可选模式：
- `path`：直接读取运行时可访问的目录（推荐，适合 Colab VSCode 插件映射路径）
- `zip_url`：通过 URL 下载 zip 并自动解压
- `drive_path`：读取 Google Drive 已存在目录（必要时自动 mount）

In [ ]:
import os
import shutil
import urllib.request
import zipfile
from pathlib import Path

# ===== 用户配置区（按需修改） =====
SOURCE_MODE = "path"          # path | zip_url | drive_path

# 1) path 模式：直接给可访问目录（不存在或不含项目结构时会自动扫描）
PROJECT_PATH = "."            # 例如：/content/<your_project_folder>

# 2) zip_url 模式：给 zip 链接（例如 GitHub Release / 对象存储）
ZIP_URL = ""

# 3) drive_path 模式：给 Drive 中项目目录
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/EGoTouchRev-rebuild"
# ==================================

WORK_DIR = Path("/content/work") if Path("/content").exists() else Path("./work")
WORK_DIR.mkdir(parents=True, exist_ok=True)


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    if (start / "build" / "config.ini").exists():
        return start
    for cfg in start.glob("**/build/config.ini"):
        return cfg.parents[1]
    raise FileNotFoundError(f"在 {start} 下未找到 build/config.ini")


def auto_discover_project_root() -> Path | None:
    probe_roots = [
        Path.cwd(),
        Path("/content"),
        Path("/workspace"),
        Path("/root"),
        Path("/kaggle/working"),
    ]
    if os.name == "nt":
        probe_roots += [Path("d:/source/repos"), Path("c:/Users")]

    seen = set()
    for root in probe_roots:
        try:
            rr = root.resolve()
        except Exception:
            continue
        if rr in seen or not rr.exists():
            continue
        seen.add(rr)
        try:
            found = find_project_root(rr)
            print(f"auto discovered from: {rr}")
            return found
        except Exception:
            pass
    return None


def print_debug_hints():
    print("\n[Hint] 请先确认你的项目目录已出现在 Colab 运行时文件系统中。")
    if Path("/content").exists():
        children = sorted([p.name for p in Path("/content").iterdir()])[:40]
        print("[Hint] /content 下当前目录:", children)
    print("[Hint] 如果你知道项目目录，直接设置：")
    print('PROJECT_PATH = "/content/<your_project_folder>"')
    print('或 SOURCE_MODE = "drive_path" 并设置 DRIVE_PROJECT_PATH')


if SOURCE_MODE == "path":
    src = Path(PROJECT_PATH)
    project_root = None

    if src.exists():
        try:
            project_root = find_project_root(src)
        except Exception:
            print(f"PROJECT_PATH 存在但不含 build/config.ini: {src}，尝试自动扫描...")

    if project_root is None:
        if not src.exists():
            print(f"PROJECT_PATH 不存在: {src}，尝试自动扫描...")
        project_root = auto_discover_project_root()

    if project_root is None:
        print_debug_hints()
        raise FileNotFoundError(
            "未找到项目根目录。请将 PROJECT_PATH 改为包含 build/config.ini 的目录，"
            "或改用 zip_url / drive_path 模式。"
        )

elif SOURCE_MODE == "zip_url":
    if not ZIP_URL:
        raise ValueError("SOURCE_MODE=zip_url 时，必须填写 ZIP_URL")
    zip_path = WORK_DIR / "project.zip"
    urllib.request.urlretrieve(ZIP_URL, zip_path)
    extract_dir = WORK_DIR / "project_unzip"
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)
    project_root = find_project_root(extract_dir)

elif SOURCE_MODE == "drive_path":
    drive_root = Path("/content/drive")
    if not drive_root.exists():
        from google.colab import drive
        drive.mount("/content/drive")
    src = Path(DRIVE_PROJECT_PATH)
    if not src.exists():
        raise FileNotFoundError(f"DRIVE_PROJECT_PATH 不存在: {src}")
    project_root = find_project_root(src)

else:
    raise ValueError(f"未知 SOURCE_MODE: {SOURCE_MODE}")

os.chdir(project_root)
print("cwd:", Path.cwd())
print("config exists:", (Path.cwd() / "build" / "config.ini").exists())
print("dvr exists:", (Path.cwd() / "build" / "exports" / "dvr").exists())

main()

PROJECT_PATH 存在但不含 build/config.ini: .，尝试自动扫描...


FileNotFoundError: 未找到项目根目录。请将 PROJECT_PATH 改为包含 build/config.ini 的目录，或改用 zip_url / drive_path 模式。